In [8]:
pip install requests selenium beautifulsoup4

  Using cached selenium-4.31.0-py3-none-any.whl.metadata (7.5 kB)
   ---------------------------------------- 0.0/9.4 MB ? eta -:--:--
   ------------------------- -------------- 6.0/9.4 MB 30.7 MB/s eta 0:00:01
   ---------------------------------------- 9.4/9.4 MB 29.0 MB/s eta 0:00:00
  Attempting uninstall: attrs
    Found existing installation: attrs 23.1.0
    Uninstalling attrs-23.1.0:
      Successfully uninstalled attrs-23.1.0
Note: you may need to restart the kernel to use updated packages.


In [9]:
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import pandas as pd
import json
import os

In [4]:
# 크롬 드라이버 생성 및 네이버 쇼핑 페이지 접속 후 로딩 대기
driver = webdriver.Chrome()
driver.maximize_window()
driver.get('https://shopping.naver.com/home')
driver.implicitly_wait(3)

In [5]:
# 검색어 입력 및 검색
search_word = '캠핑 숯'
search_box = driver.find_element(By.XPATH, '//input[contains(@class, "searchInput_search_text")]')
search_box.clear()
search_box.send_keys(search_word)
search_box.send_keys(Keys.ENTER)

# 페이지 로딩이 완료될 때까지 대기
driver.implicitly_wait(3)

In [6]:
# 스크롤을 사용해서 더 많은 상품 로드
last_height = driver.execute_script("return document.body.scrollHeight")

# 지정한 횟수만큼 스크롤 내림
scroll_count = 1
for i in range(scroll_count):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    driver.implicitly_wait(1)
    # 스크롤 내린 후 높이 측정
    new_height = driver.execute_script("return document.body.scrollHeight")
    # 더 이상 로드되는 내용이 없으면 종료
    if new_height == last_height:
        break
    
    last_height = new_height

# 페이지 소스 가져오기 및 BeautifulSoup 파싱
page_source = driver.page_source
soup = BeautifulSoup(page_source, "html.parser")

In [11]:
# XPath를 사용해서 상품 정보 가져오기
items_path = '//*[@id="composite-card-list"]/div/ul[1]/li'
items = driver.find_elements(By.XPATH, items_path)
items

[<selenium.webdriver.remote.webelement.WebElement (session="1b5df6c159d9977ce14a0761954cd13e", element="f.2F9B7892BB0492710A600FBBA20F0D5F.d.503E474986EED69D713D99D3D809C296.e.65755")>,
 <selenium.webdriver.remote.webelement.WebElement (session="1b5df6c159d9977ce14a0761954cd13e", element="f.2F9B7892BB0492710A600FBBA20F0D5F.d.503E474986EED69D713D99D3D809C296.e.65756")>,
 <selenium.webdriver.remote.webelement.WebElement (session="1b5df6c159d9977ce14a0761954cd13e", element="f.2F9B7892BB0492710A600FBBA20F0D5F.d.503E474986EED69D713D99D3D809C296.e.65757")>,
 <selenium.webdriver.remote.webelement.WebElement (session="1b5df6c159d9977ce14a0761954cd13e", element="f.2F9B7892BB0492710A600FBBA20F0D5F.d.503E474986EED69D713D99D3D809C296.e.65758")>,
 <selenium.webdriver.remote.webelement.WebElement (session="1b5df6c159d9977ce14a0761954cd13e", element="f.2F9B7892BB0492710A600FBBA20F0D5F.d.503E474986EED69D713D99D3D809C296.e.65759")>,
 <selenium.webdriver.remote.webelement.WebElement (session="1b5df6c159

In [17]:
# 상품 정보를 가져오면서 에러가 발생한 경우 예외처리 후 생략
products = []

n = 1
for item in items:
    print(f"{n}번째 상품: ", end="")
    try:
        item_a_tag = item.find_element(By.XPATH, './/div/a[contains(@class, "basicProductCard")]')
        info = item_a_tag.get_attribute('data-shp-contents-dtl')
        info_dict = json.loads(info)
        product_name = next(item['value'] for item in info_dict if item['key'] == 'prod_nm')
        product_price = next(item['value'] for item in info_dict if item['key'] == 'price')
        product_link = item_a_tag.get_attribute('href')
        products.append({
            '상품명': product_name,
            '가격': product_price,
            '링크': product_link
        })
        print(info)
    except NoSuchElementException:
        print("못찾음")
    n += 1

1번째 상품: [{"key":"chnl_prod_no","value":"5969670976"},{"key":"nv_mid","value":"83514170464"},{"key":"click_url","value":"https://smartstore.naver.com/main/products/5969670976"},{"key":"price","value":"27500"},{"key":"cat_id","value":"50002665"},{"key":"loungeonly_yn","value":"n"},{"key":"purchase_yn","value":"n"},{"key":"promotion_yn","value":"n"},{"key":"coupontag_yn","value":"n"},{"key":"nfa_type","value":"오늘출발"},{"key":"prod_nm","value":"[ 캠핑홀릭 숯장군 (벌크상품) ] 국내산 세계유일 7번구운 참숯 1등급 백탄"}]
2번째 상품: [{"key":"chnl_prod_no","value":"11390235607"},{"key":"nv_mid","value":"88934745972"},{"key":"click_url","value":"https://smartstore.naver.com/main/products/11390235607"},{"key":"price","value":"29000"},{"key":"cat_id","value":"50002665"},{"key":"loungeonly_yn","value":"n"},{"key":"purchase_yn","value":"n"},{"key":"promotion_yn","value":"n"},{"key":"coupontag_yn","value":"n"},{"key":"nfa_type","value":"오늘출발"},{"key":"prod_nm","value":"수제천 비장탄 커피나무 커피숯 참숯 캠핑용 업소용 10kg"}]
3번째 상품: [{"key":"chnl_prod_

In [18]:
products

[{'상품명': '[ 캠핑홀릭 숯장군 (벌크상품) ] 국내산 세계유일 7번구운 참숯 1등급 백탄',
  '가격': '27500',
  '링크': 'https://ader.naver.com/v1/KYdCsPQUmi2vD9Cba_IycRhRHMCvDnrxR9DBiRTlXaA6hB0xYm8vyw2C-8iwGS-ZD-Z6s1DH2ujuS8p5nv8tVnIkmeof9aXspeEGT7oBMpzRS7f-k6aJjH-d-k_wIiW8azdc_FtG2ygGaBd3d_9WTQzJ4A8uDjrK0zLVU_ywYKhAMYGaHpkY8kaM7DUz_Y1svrRqVT1KArr8orhI9APHWNhnWMf883eh1Cyg3Qyl3U7DH1EK99fexThDNuGuAwswok5MChvsI9XZQYNMuyNu99UN6BQXyLgh4tZgQjKwa57FSQociMrDYUlXOZNFhBQii6TWYK_sm9ZaUuu0eH54NYsMD3EyQYb_8bKcGEfSUSt3zR5JDzV23vNaSmfpwX2uZlrZ4jvl1BtS8P4awbceR1DT2K1UpxebF-L4M7MpLIC19tf99HDQyQsfEfhMMC_H2Ucb7b3zJsFCgRz9FOAMKG4MMJ0lE4KnmGd2Z6ZZne4=?c=pc.nplusstore.npla&t=0'},
 {'상품명': '수제천 비장탄 커피나무 커피숯 참숯 캠핑용 업소용 10kg',
  '가격': '29000',
  '링크': 'https://ader.naver.com/v1/3tyIt3S-T97Ocp0Y2RJsvsi9gZTQLfxWYMtzRF3YWsKLODZ4HALD3E5rN5hK890LJIxs0V3CEzw_UpVIKTr_wBf5Mu4fO3g4-nFyfDY_6vGKPK3ULZTedJUjxyWqIAgK3xwN9kPEtMZMEOiLF3WB_rYT1D4nd7EcxNYZ4hi2idw0WSdXIpu-mILmGFopMrUoGr4GFIrh6IbQ6DYijmwFKLfW2WHLSepGH0P4RjeKqsgJFYyf1HvBVCk0Zgkb_r6i5FBtgXzWF91CJB6fU0

In [19]:
# 엑셀 파일 경로 지정
path_to_save = './data'
file_name = "naver_shopping_camping_charcoal.xlsx"
file_path = os.path.join(path_to_save, file_name)

# 지정한 경로가 존재하지 않으면 경로 생성
if not os.path.exists(path_to_save):
    os.makedirs(path_to_save)
    print(f"경로 {path_to_save}가 생성되었습니다.")

# DataFrame으로 변환 후 저장
df = pd.DataFrame(products)
df.to_excel(file_path, index=False)
print(f"엑셀 파일이 {file_path}에 저장되었습니다.")

# WebDriver 종료
driver.quit()

엑셀 파일이 ./data/naver_shopping_camping_charcoal.xlsx에 저장되었습니다.
